# Основы машинного обучения, ИИМУП

## НИУ ВШЭ, 2024-25 учебный год

# Домашнее задание 3: Рекуррентные нейронные сети

Задание выполнил(а):

    (впишите свои фамилию и имя)

## Общая информация

__Внимание!__  


* Домашнее задание выполняется самостоятельно
* Не допускается помощь в решении домашнего задания от однокурсников или третьих лиц. «Похожие» решения считаются плагиатом, и все задействованные студенты — в том числе и те, у кого списали, — не могут получить за него больше 0 баллов
* Использование в решении домашнего задания генеративных моделей (ChatGPT и так далее) за рамками справочной и образовательной информации для генерации кода задания — считается плагиатом, и такое домашнее задание оценивается в 0 баллов
* Старайтесь сделать код как можно более оптимальным. Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

### О задании

В этом задании вам предстоит самостоятельно реализовать модель LSTM для решения задачи классификации с пересекающимися классами (multi-label classification). Это вид классификации, в которой каждый объект может относиться одновременно к нескольким классам. Такая задача часто возникает при классификации фильмов по жанрам, научных или новостных статей по темам, музыкальных композиций по инструментам и так далее.

В нашем случае мы будем работать с датасетом биотехнических новостей и классифицировать их по темам. Этот датасет уже предобработан: текст приведен к нижнему регистру, удалена пунктуация, все слова разделены проблелом.

In [7]:
import pandas as pd

In [8]:
BIOTECH_NEWS = 'https://www.dropbox.com/scl/fi/v624fdeg3bikyxyvr93i3/biotech_news.tsv?rlkey=iaekqfwtbqswl8vd5dpnd9d92&st=mzgbw5km&dl=1'

In [9]:
dataset = pd.read_csv(BIOTECH_NEWS, sep='\t')
dataset.head()

,text,labels
0,drive your plow over the bones of the dead by ...,other
1,in the recently tabled national budget denel h...,other
2,shares take a break its good for you picture g...,other
3,reso is currently hiring for two positions pro...,other
4,charter buyer club what is the charter buyer c...,other


## Предобработка лейблов


__Задание 1 (1.5 балла)__. Как вы можете заметить, лейблы записаны в виде строк, разделенных запятыми. Для работы с ними нам нужно преобразовать их в числа. Так как каждый объект может принадлежать нескольким классам, закодируйте лейблы в виде векторов из 0 и 1, где 1 означает, что объект принадлежит соответствующему классу, а 0 – не принадлежит. Имея такую кодировку, мы сможем обучить модель, решая задачу бинарной классификации для каждого класса.

In [11]:
from sklearn.preprocessing import MultiLabelBinarizer

labels_split = dataset["labels"].apply(lambda x: x.split(","))
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(labels_split)

label_names = mlb.classes_
y.shape

(3039, 55)

## Предобработка данных

В этом задании мы будем обучать рекуррентные нейронные сети. Как вы знаете, они работают лучше для коротких текстов, так как не очень хорошо улавливают далекие зависимости. Для уменьшение длин текстов их стоит почистить.

Сразу разделим выборку на обучающую и тестовую, чтобы считать все нужные статистики только по обучающей.

In [14]:
from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test = train_test_split(
    dataset["text"].values,
    y,
    test_size=0.2,  # do not change this
    random_state=0)  # do not change this


__Задание 2 (1.5 балла)__. Удалите из текстов стоп слова, слишком редкие и слишком частые слова. Гиперпараметры подберите самостоятельно (в идеале их стоит подбирать по качеству на тестовой выборке). Если вы считаете, что стоит добавить еще какую-то обработку, то сделайте это. Важно не удалить ничего, что может повлиять на предсказание класса.

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords")


stop_words = list(stopwords.words("english"))
vectorizer = CountVectorizer(min_df=3, max_df=0.8, stop_words=stop_words)

vectorizer.fit(texts_train)

valid_tokens = set(vectorizer.get_feature_names_out())

def preprocess_text(text):
    tokens = text.split()
    tokens = [token for token in tokens if token in valid_tokens]
    return " ".join(tokens)

texts_train = [preprocess_text(t) for t in texts_train]
texts_test = [preprocess_text(t) for t in texts_test]# your code here

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/andrejkazancev/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


__Задание 3 (2 балла)__. Осталось перевести тексты в индексы токенов, чтобы их можно было подавать в модель. У вас есть две опции, как это сделать:
1. __(+0 баллов)__ Токенизировать тексты по словам.
2. __(до +5 баллов)__ Реализовать свою токенизацию BPE. Количество баллов будет варьироваться в зависимости от эффективности реализации. При реализации нельзя пользоваться специализированными библиотеками.

Токенизируйте тексты, переведите их в списки индексов и сложите вместе с лейблами в `DataLoader`. Не забудьте добавить в `DataLoader` `collate_fn`, которая будет дополнять все короткие тексты в батче паддингами. Для маппинга токенов в индексы вам может пригодиться `gensim.corpora.dictionary.Dictionary`.

In [37]:
from collections import defaultdict, Counter

# Подготовка данных
corpus = texts_train.copy()
corpus = [word for sentence in corpus for word in sentence.strip().split()]
vocab = Counter(corpus)

# Представим каждое слово как список символов + спецсимвол конца слова
def word_to_symbols(word):
    return list(word) + ["</w>"]

tokenized_vocab = {" ".join(word_to_symbols(word)): count for word, count in vocab.items()}

# Функция для подсчета частот пар символов
def get_pair_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq
    return pairs

# Слияние самой частой пары
def merge_pair(pair, vocab):
    merged_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        merged_vocab[new_word] = vocab[word]
    return merged_vocab

# Итеративный BPE
def learn_bpe(vocab, num_merges=1000):
    merges = []
    for _ in range(num_merges):
        pair_stats = get_pair_stats(vocab)
        if not pair_stats:
            break
        most_common = max(pair_stats, key=pair_stats.get)
        vocab = merge_pair(most_common, vocab)
        merges.append(most_common)
    return merges

# Запуск
bpe_merges = learn_bpe(tokenized_vocab, num_merges=1000)
print("Пример merge-правил:", bpe_merges[:10])

Пример merge-правил: [('s', '</w>'), ('e', '</w>'), ('i', 'n'), ('e', 'r'), ('d', '</w>'), ('t', '</w>'), ('o', 'n'), ('e', 'n'), ('y', '</w>'), ('a', 'l')]


In [39]:
def apply_bpe(word, bpe_merges):
    word = list(word) + ['</w>']
    while True:
        pairs = [(word[i], word[i+1]) for i in range(len(word)-1)]
        merge_candidates = {pair: idx for idx, pair in enumerate(bpe_merges) if pair in pairs}
        if not merge_candidates:
            break
        # выбрать первое по порядку в списке merge-правил
        best_pair = min(merge_candidates, key=merge_candidates.get)
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == best_pair:
                new_word.append(word[i] + word[i+1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        word = new_word
    return word

In [41]:
def tokenize_bpe_texts(texts, bpe_merges):
    tokenized_texts = []
    for text in texts:
        tokens = []
        for word in text.split():
            tokens.extend(apply_bpe(word, bpe_merges))
        tokenized_texts.append(tokens)
    return tokenized_texts

In [43]:
tokenized_train = tokenize_bpe_texts(texts_train, bpe_merges)
tokenized_test = tokenize_bpe_texts(texts_test, bpe_merges)

In [44]:
from collections import Counter

# Соберём частотный словарь всех токенов
token_counts = Counter(token for text in tokenized_train for token in text)

# Спецтокены:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Стартуем словарь с этих токенов
token2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token in token_counts:
    token2idx[token] = len(token2idx)

In [45]:
def encode(text_tokens, token2idx):
    return [token2idx.get(token, token2idx[UNK_TOKEN]) for token in text_tokens]

X_train = [encode(text, token2idx) for text in tokenized_train]
X_test = [encode(text, token2idx) for text in tokenized_test]

In [49]:
import torch
import numpy as np
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs = [torch.tensor(x, dtype=torch.long) for x in inputs]
    targets = np.array(targets)
    targets = torch.as_tensor(targets, dtype=torch.float32)

    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=token2idx["<PAD>"])
    return padded_inputs, targets

In [50]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]

In [53]:
from torch.utils.data import DataLoader

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

## Метрика качества

Перед тем, как приступить к обучению, нам нужно выбрать метрику оценки качества. Так как в задаче классификации с пересекающимися классами классы часто несбалансированы, чаще всего в качестве метрики берется [F1 score](https://en.wikipedia.org/wiki/F-score).

Функция `compute_f1` принимает истинные метки и предсказанные и считает среднее значение F1 по всем классам. Используйте ее для оценки качества моделей.

$$
F1_{total} = \frac{1}{K} \sum_{k=1}^K F1(Y_k, \hat{Y}_k),
$$
где $Y_k$ – истинные значения для класса k, а $\hat{Y}_k$ – предсказания.

In [55]:
from sklearn.metrics import f1_score

def compute_f1(y_true, y_pred):
    assert y_true.ndim == 2
    assert y_true.shape == y_pred.shape

    return f1_score(y_true, y_pred, average='macro')

## Обучение моделей

### RNN

В качестве бейзлайна обучим самую простую рекуррентную нейронную сеть. Напомним, что блок RNN выглядит таким образом.

<img src="https://i.postimg.cc/yYbNBm6G/tg-image-1635618906.png" alt="drawing" width="400"/>

Его скрытое состояние обновляется по формуле
$h_t = \sigma(W x_{t} + U h_{t-1} + b_h)$. А предсказание считается с помощью применения линейного слоя к последнему токену
$o_T = V h_T + b_o$. В качестве функции активации выберите гиперболический тангенс.

__Задание 4 (2 балла)__. Реализуйте RNN в соответствии с формулой выше и обучите ее на нашу задачу. Нулевой скрытый вектор инициализируйте нулями, так модель будет обучаться стабильнее, чем при случайной инициализации. После этого замеряйте качество на тестовой выборке. У вас должно получиться значение F1 не меньше 0.33, а само обучение не должно занимать много времени.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# Простейшая RNN модель вручную
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.hidden_dim = hidden_dim
        self.rnn = nn.Tanh()
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        emb = self.embedding(x)  # (batch, seq, emb_dim)
        h = torch.zeros(x.size(0), self.hidden_dim, device=x.device)
        for t in range(x.size(1)):
            h = self.rnn(emb[:, t, :] @ W + h @ U + b)  # вручную обновляем
        out = self.linear(h)
        return torch.sigmoid(out)

# Упрощённо: зададим параметры и матрицы вручную для h update
# Эти параметры можно встроить в модель, но для прозрачности — выносим отдельно
embedding_dim = 64
hidden_dim = 64
output_dim = y_train.shape[1]  # многоклассовый вывод
vocab_size = len(token2idx)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleRNN(vocab_size, embedding_dim, hidden_dim, output_dim).to(device)
W = nn.Parameter(torch.randn(embedding_dim, hidden_dim, device=device) * 0.1)
U = nn.Parameter(torch.randn(hidden_dim, hidden_dim, device=device) * 0.1)
b = nn.Parameter(torch.zeros(hidden_dim, device=device))

optimizer = Adam([*model.parameters(), W, U, b], lr=0.001)
loss_fn = nn.BCELoss()

def train_rnn(model, loader):
    model.train()
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device).float()
        preds = model(batch_x)
        loss = loss_fn(preds, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate_rnn(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            preds = model(batch_x).cpu()
            all_preds.append(preds)
            all_targets.append(batch_y)
    y_pred = torch.cat(all_preds).round().numpy()
    y_true = torch.cat(all_targets).numpy()
    return compute_f1(y_true, y_pred)

# Обучим
for epoch in range(10):
    train_rnn(model, train_loader)
    f1 = evaluate_rnn(model, test_loader)
    print(f"Epoch {epoch}: F1 = {f1:.4f}")# your code here

/var/folders/f2/r5hh7nns2dd2_jhmznc8ntp80000gn/T/ipykernel_11758/4199493880.py:7: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  targets = torch.tensor(targets, dtype=torch.float32)
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 0: F1 = 0.0000


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1: F1 = 0.0000


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2: F1 = 0.0000


### LSTM

<img src="https://i.postimg.cc/pL5LdmpL/tg-image-2290675322.png" alt="drawing" width="400"/>

Теперь перейдем к более продвинутым рекурренным моделям, а именно LSTM. Из-за дополнительного вектора памяти эта модель должна гораздо лучше улавливать далекие зависимости, что должно напрямую отражаться на качестве.

Параметры блока LSTM обновляются вот так ($\sigma$ означает сигмоиду):
\begin{align}
f_{t} &= \sigma(W_f x_{t} + U_f h_{t-1} + b_f) \\
i_{t} &= \sigma(W_i x_{t} + U_i h_{t-1} + b_i) \\
\tilde{c}_{t} &= \tanh(W_c x_{t} + U_c h_{t-1} + b_i) \\
c_{t} &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
o_{t} &= \sigma(W_t x_{t} + U_t h_{t-1} + b_t) \\
h_t &= o_t \odot \tanh(c_t)
\end{align}

__Задание 5 (2 балла).__ Реализуйте LSTM по описанной схеме. Выберите гиперпараметры LSTM так, чтобы их общее число (без учета слоя эмбеддингов) примерно совпадало с числом параметров обычной RNN, но размерность скрытого слоя была не меньше 64. Так мы будем сравнивать архитектуры максимально независимо. Обучите LSTM до сходимости и сравните качество с RNN на тестовой выборке. Удалось ли получить лучший результат? Как вы можете это объяснить?

In [24]:
# your code here

__Задание 6 (1 балл).__ В этом задании у вас есть две опции на выбор: добавить __двунаправленность__ для LSTM _или_ добавить __многослойность__. Можно сделать и то, и другое, но дополнительных баллов за это мы не дадим, только бесконечный респект. Обе модификации реализуются довольно просто (буквально 4 строчки кода, если вы аккуратно реализовали модель) и дают примерно одинаковый прирост в качестве. Сделайте выводы: стоит ли увеличивать размер модели в несколько раз?

In [26]:
# your code here